In [1]:
import os
import time
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import VCF_parser

In [2]:
# Global vars
wd = '/stornext/Bioinf/data/bioinf-data/Papenfuss_lab_projects/SVensemble2/vcf_parsing'
gridss_path = f'{wd}/data/test_gridss_output.NA12878.vcf'
gnomad_path = '/home/users/allstaff/shikov.v/projects/SVensemble2/data/gnomad/4.1.0/downloads/gnomad.v4.1.sv.sites.vcf.gz'

In [3]:
test_mod, test_mode = VCF_parser.get_read_params(gridss_path)

In [4]:
test_mod

<module 'builtins' (built-in)>

In [5]:
def split_vcf_by_chrom(vcf_df, chrom_list=None):
    if chrom_list is not None:
        return {chrom: vcf_df.loc[vcf_df.CHROM == chrom] \
                for chrom in chrom_list}
    return {chrom: gnomad_vcf_df.loc[gnomad_vcf_df.CHROM == chrom] \
                for chrom in pd.unique(vcf_df.CHROM)}

In [6]:
def generate_truth_list(vcf_df, ref_df_dict, report_freq=100000):

    truth_list = []
    pos_series = vcf_df.POS
    chrom_series = vcf_df.CHROM
    
    if set(ref_df_dict.keys()) != set(chrom_series.unique()):
        raise KeyError('Chromosomes in prediction dataset do not match chromosomes in gnomad')
    
    for i, (pos, chrom) in enumerate(zip(pos_series, chrom_series)):
        # print(f'{chrom}: {pos}')
        # overlapping_ref_positions = (gnomad_df[chrom].POS > pos - 25) & (gnomad_df[chrom].POS < pos + 25)
    
        truth_list.append(
            ((ref_df_dict[chrom].POS > pos - 25) & (ref_df_dict[chrom].POS < pos + 25)).any()
        )
        
        if i % report_freq == 0:
            print(f'processed {i} entries')

    return truth_list

In [7]:
gnomad_vcf_df = VCF_parser.parse_vcf(gnomad_path, line_limit=500000)

Processed 0 entries                 Elapsed time: 0.030124187469482422 seconds.
Processed 100000 entries                 Elapsed time: 34.149534463882446 seconds.
Processed 200000 entries                 Elapsed time: 69.98905372619629 seconds.
Processed 300000 entries                 Elapsed time: 105.60838150978088 seconds.
Processed 400000 entries                 Elapsed time: 140.9914288520813 seconds.
Processed 500000 entries                 Elapsed time: 176.18842148780823 seconds.
Finished processing. Processed 500000 entries out of 500000. Elapsed time: 239.9264633655548 seconds.


In [8]:
gridss_vcf_df = VCF_parser.parse_vcf(gridss_path)

Processed 0 entries                 Elapsed time: 0.009137868881225586 seconds.
Processed 100000 entries                 Elapsed time: 5.46272611618042 seconds.
Processed 200000 entries                 Elapsed time: 10.919637441635132 seconds.
Processed 300000 entries                 Elapsed time: 16.406007289886475 seconds.
Processed 400000 entries                 Elapsed time: 21.87128496170044 seconds.
Processed 500000 entries                 Elapsed time: 27.40699052810669 seconds.
Processed 600000 entries                 Elapsed time: 33.01365089416504 seconds.
Processed 700000 entries                 Elapsed time: 38.53498554229736 seconds.
Processed 800000 entries                 Elapsed time: 44.06725788116455 seconds.
Processed 900000 entries                 Elapsed time: 49.60082411766052 seconds.
Processed 1000000 entries                 Elapsed time: 55.10003733634949 seconds.
Finished processing. Processed 1007409 entries out of 1007410. Elapsed time: 81.83646869659424 sec

In [59]:
gnomad_vcf_df = split_vcf_by_chrom(gnomad_vcf_df, pd.unique(gridss_vcf_df.CHROM))

In [64]:
truth_list = generate_truth_list(gridss_vcf_df, gnomad_vcf_df)
gridss_vcf_df['MATCH_FOUND'] = truth_list

processed 0 entries
processed 100000 entries
processed 200000 entries
processed 300000 entries
processed 400000 entries
processed 500000 entries
processed 600000 entries
processed 700000 entries
processed 800000 entries
processed 900000 entries
processed 1000000 entries


In [66]:
gnomad_vcf_df = pd.concat(gnomad_vcf_df, ignore_index=True)
# gridss_vcf_df = pd.concat(gridss_vcf_df, ignore_index=True)

In [67]:
gnomad_vcf_df = gnomad_vcf_df.sort_values(by=['POS'])
gridss_vcf_df = gridss_vcf_df.sort_values(by=['POS'])

In [31]:
gridss_vcf_df = pd.merge_asof(gridss_vcf_df, gnomad_vcf_df, on='POS', by='CHROM', direction='nearest')

In [49]:
gridss_vcf_df.iloc[0:400,:].to_csv('~/projects/SVensemble2/truth_set_gen/saved/test_dataset.tsv', sep='\t')

In [55]:
test_df = gridss_vcf_df.iloc[0:400,:]

In [56]:
test_df = pd.concat([test_df[['CHROM', 'ID_x', 'REF_x']], test_df.select_dtypes(include=np.number), test_df[['MATCH_FOUND']]], axis=1)

In [57]:
test_df.to_csv('~/projects/SVensemble2/truth_set_gen/saved/test_dataset.filtered.tsv', sep='\t')